In [1]:
import cma
import numpy as np
import dill as pkl
from scipy.optimize import minimize
from F2_diff_solver import prepare_costfun, prepare_stopping_costfun

from mat4py import loadmat

In [2]:
X_RANGE = (0.0, 20.0)
U_RANGE = (10.0, 100.0)
U_SIZE = 10
X1_SIZE = 11

f_data = loadmat('Frantisek_Task1/data_file.mat')
# which time & output response data vector & corresp. coef. for unit input step signal
ts = {opening: np.array(f_data['T{key}'.format(key=opening)]).flatten() for opening in range(10,101,10)}
yms = {opening: np.array(f_data['Y{key}'.format(key=opening)]).flatten() for opening in range(10,101,10)}

A1_tile = np.array([151.57, 89.4135, 36.847, 17.458, 10.618, 7.9709, 6.1632, 4.7546, 4.1182, 3.8926])
A2_tile = np.array([0.001, 12.9696, 159.9975, 48.7086, 20.6098, 12.1711, 8.0068, 4.448, 4.2918, 3.8111])
k_tile = np.array([1.235, 1.019, 0.693, 0.528, 0.417, 0.347, 0.3, 0.259, 0.231, 0.209])
Td_tile = np.array([0.1, 0.1, 0.1, 0.34, 0.55, 0.54, 0.52, 0.53, 0.41, 0.41])

A1_grid = np.tile(A1_tile, (X1_SIZE, 1))
A2_grid = np.tile(A2_tile, (X1_SIZE, 1))
k_grid = np.tile(k_tile, (X1_SIZE, 1))

A1_bounds = (0.0, 200.0)
A2_bounds = (0.0, 200.0)
k_bounds = (0.0, 2.0)
Td_bounds = (0.0, 5.0)

In [3]:
parameters0_genome = np.concatenate([A1_grid.flatten(), A2_grid.flatten(), k_grid.flatten(), Td_tile])
l_bounds = [A1_bounds[0]] * X1_SIZE * U_SIZE + [A2_bounds[0]] * X1_SIZE * U_SIZE + [k_bounds[0]] * X1_SIZE * U_SIZE + [Td_bounds[0]] * U_SIZE
u_bounds = [A1_bounds[1]] * X1_SIZE * U_SIZE + [A2_bounds[1]] * X1_SIZE * U_SIZE + [k_bounds[1]] * X1_SIZE * U_SIZE + [Td_bounds[1]] * U_SIZE

xbest=np.array([151.56548154557015, 89.42798683850181, 36.845858997305804, 17.45951239595446, 10.616180110884176, 7.973564574270716, 6.1731866153137975, 4.754531209977465, 4.116788408646192, 3.8987615386846413, 151.57026781102212, 89.40179183892006, 36.84759776155948, 17.46329132832531, 10.611143887655908, 7.9799422908357345, 6.15358687370448, 4.74799480244382, 4.116366061496153, 3.8747748034928935, 151.58920589876575, 89.41886597084866, 36.847022290499865, 17.46398675205143, 10.633958012428693, 7.968320742253536, 6.1569247416513715, 4.762234308610431, 4.116737997044376, 3.8967838990305843, 151.57041748400505, 89.40431204290167, 36.85818690436538, 17.450735576191963, 10.617960303301237, 7.961472224573884, 6.174328233299203, 4.770387218133122, 4.105739847624881, 3.8922542891014675, 151.56675984382633, 89.39703787039265, 36.853444179428756, 17.464246035015027, 10.613373163886436, 7.972301329406726, 6.161784324677308, 4.766969589548705, 4.123395314375281, 3.870032380613614, 151.57719515537903, 89.41502082816388, 36.849183778142844, 17.458283403519072, 10.62185051158933, 7.981467191951256, 6.160980078489406, 4.762829793658321, 4.120358063345236, 3.8887609964169143, 151.57085694772286, 89.41504678991622, 36.84557860439572, 17.467426968599987, 10.625962642326812, 7.97541200501556, 6.166661968078572, 4.7661103000065, 4.126969505795368, 3.8900015877315233, 151.55937670496417, 89.4178371469164, 36.83824823557482, 17.457762629921827, 10.62540877847744, 7.971016201326983, 6.164320204771334, 4.757141894103702, 4.114193091547604, 3.8885239843825854, 151.57305698748362, 89.41248840858724, 36.85787502309875, 17.453352876901953, 10.623269358350338, 7.9721853499899975, 6.154561136714278, 4.749787347417263, 4.11910714813554, 3.8937232170500615, 151.5678451554599, 89.41335430220636, 36.846221467494125, 17.463172353330876, 10.619814483766184, 7.962279204381267, 6.170981938540459, 4.739652986105401, 4.120414589427169, 3.8905865758575353, 151.57438377626252, 89.40578688062196, 36.84226556575196, 17.46265786992095, 10.622598774706617, 7.972043120943939, 6.172920128431466, 4.746479911432168, 4.109223531962492, 3.8919503342456654, 0.0002699045509381398, 12.990354941724078, 160.01232542669985, 48.718513510314835, 20.607148772975407, 12.163921378871294, 8.010895928569704, 4.434712361604848, 4.279788588738883, 3.8160396800126524, 0.0011733127668588177, 12.972183703629435, 159.99830863430586, 48.71314695496228, 20.610509874116662, 12.174289548499917, 7.999946720669938, 4.438306015937744, 4.289967183955062, 3.815531353167354, 0.0010317306556019373, 12.982951485216867, 159.99844616964077, 48.697761470031686, 20.609305529637034, 12.159994212116205, 8.002623730468832, 4.452918144853825, 4.299705207476746, 3.7995192933016826, 0.0006132932055808975, 12.969538249258333, 159.98893447300148, 48.7130493028169, 20.60611460222514, 12.172234285613547, 8.002590357966659, 4.460860215972145, 4.29106202286296, 3.8197946637122886, 0.0014202405380397058, 12.956798207966084, 159.99984400890375, 48.70392004746577, 20.606808818128954, 12.190942253263536, 8.012027203552417, 4.446746478724704, 4.284736121169683, 3.820503364990582, 4.4918223028509424e-05, 12.967334909423087, 159.98737045010063, 48.7179165950414, 20.604757319078892, 12.165678928105596, 8.00027279523433, 4.444720484404772, 4.294350936598218, 3.813335615152473, 0.001542785656791527, 12.96247360845056, 159.99313498295427, 48.70031626936148, 20.614632533162958, 12.171869661054359, 8.014898717780364, 4.452133561582149, 4.292815855943007, 3.8208063380872472, 0.0010428395369774263, 12.973554492056087, 159.99561018334919, 48.701162457357746, 20.60888964437967, 12.160044999468303, 8.005116075040075, 4.446851373259145, 4.2904726281923375, 3.8136076232107112, 0.0012626426151762854, 12.975575172103932, 159.9958157379324, 48.708888072713115, 20.598141530654132, 12.177433815709994, 8.007050665237832, 4.43946921113146, 4.268843066548264, 3.8053287463841183, 0.00033147430378432474, 12.951306812991891, 159.99591059686028, 48.71881840464874, 20.608893921370758, 12.166181997071298, 7.993888237976682, 4.452634604957827, 4.276070606215276, 3.8131192683655035, 0.0004000311078394222, 12.961552591913291, 160.00005430468804, 48.7053100467785, 20.625854222729185, 12.161245380154496, 7.99870473479188, 4.445045734183669, 4.296446566235445, 3.8057700039625413, 1.2394475649371917, 1.0307091624825557, 0.7135615436744216, 0.5412580185589829, 0.4309609839207744, 0.35677976297936276, 0.31985310246749055, 0.27422958460588864, 0.24192291209275757, 0.21839471081103634, 1.2416867545299952, 1.0262998180826652, 0.6937301459912594, 0.5336221836111423, 0.4210993614957136, 0.3514649071637716, 0.30948376680334255, 0.2668115904672814, 0.24273806604732925, 0.21056534717200195, 1.2460770878856773, 1.031749821208132, 0.6814503812723244, 0.5020628840311978, 0.4065818224137359, 0.32729316311640966, 0.28740124047777343, 0.24694108271582438, 0.2227316154651024, 0.19757088073126508, 1.256467977780157, 1.022251234622724, 0.6959035792352989, 0.5174717588199707, 0.4087932248189615, 0.3498794317078982, 0.28697513859731094, 0.24285712031323772, 0.22269432339310327, 0.21035632796736461, 1.2352869874472672, 1.0193349796322417, 0.6949732732575136, 0.5233067316911652, 0.4032525528821363, 0.3425771172485741, 0.2730321267376404, 0.2422081773334268, 0.23039802269424176, 0.2124143748862368, 1.2155446576096223, 1.036285764198367, 0.6809425437095971, 0.51268468252535, 0.4148609361119737, 0.3304220711257654, 0.2988385708284818, 0.24249721772357768, 0.21933614059081663, 0.20160349111500572, 1.2665266749338318, 1.005897813283808, 0.6835619111064564, 0.5353058601427115, 0.4212819077281335, 0.335524246171214, 0.29360618787988124, 0.2607225578687654, 0.2254719691787183, 0.20854583839683985, 1.2426231809691053, 1.0033495777700234, 0.6962100356263109, 0.5315168423075249, 0.4275609934347306, 0.3568254782625638, 0.3010568804577267, 0.27106099473632944, 0.2278931514071602, 0.2064391402435843, 1.2401647706043, 1.0134874373850373, 0.7073391618012367, 0.5242921604577508, 0.4194214584495984, 0.3554731599923836, 0.30535045879042283, 0.26482782646331227, 0.24434166258370785, 0.2149312289406691, 1.2152394789495626, 1.0239329127046592, 0.7220729760067321, 0.548310769812949, 0.41933868248334694, 0.35515597218757666, 0.3124083985852974, 0.2645533974786023, 0.23758210408452432, 0.21100939468978014, 1.2481000915210347, 1.0165786584722563, 0.6791145314648179, 0.5201639764650648, 0.4165154972383233, 0.34660464794355916, 0.2997939858415211, 0.2588083227680082, 0.2318714244328642, 0.2094900662995841, 0.09203736859039711, 0.10016465401720936, 0.09341619056090626, 0.3264940152236975, 0.539292851413177, 0.5317306330334961, 0.4987564158386479, 0.5074778606878402, 0.39615762231399426, 0.409994317713924])
new_xbest_scaled = np.array([0.7627963935616859, 0.4610025974587532, 0.18317552336198922, 0.08879745397764567, 0.06233211116541434, 0.028195296188350698, 0.023580253302054287, 0.02135517587624042, 0.01811120121530812, 0.020449074466899288, 0.7517423514262991, 0.43579898904205844, 0.1834640401417262, 0.07653060563046957, 0.05047945104093544, 0.04442790226945219, 0.0346906811728036, 0.022177238186051797, 0.020022025044076897, 0.0157397299102144, 0.7495283420383095, 0.4441226003547671, 0.19657807910295882, 0.09898868789430555, 0.06477142206480838, 0.05304096506279408, 0.043181564718844136, 0.03183876470917654, 0.027200746447452028, 0.018269065470613802, 0.7528231912659334, 0.4443227370474107, 0.18024956303927597, 0.0964948720997955, 0.05933416120645233, 0.04676061595499191, 0.03672394491603833, 0.034925743854057466, 0.020504831317451778, 0.0204915630743904, 0.7592201255051898, 0.4415615803641944, 0.1848772154381834, 0.0956108571997399, 0.05572163531343585, 0.044053743030601784, 0.03473257801430138, 0.023347982284242324, 0.020088371764277937, 0.021332279503033746, 0.7678254770727849, 0.44869744706514697, 0.1768100884310017, 0.08375699551092754, 0.05156219937317925, 0.03879630297651615, 0.032686477299771526, 0.025411301724645893, 0.020002388028959702, 0.022093038719000185, 0.7546347937186286, 0.4474003928089042, 0.19006208104782374, 0.07861756206393464, 0.04917121653590717, 0.04241715241955471, 0.03196184447773578, 0.021306183708910166, 0.020433658284933667, 0.018890992202267733, 0.7592572207810501, 0.4583713412379957, 0.1897924414839767, 0.08963342821662575, 0.04801853951639939, 0.03516990151882178, 0.02922244650987451, 0.024198820084710478, 0.021921416502770786, 0.01809452849771029, 0.7616270103219395, 0.44713116660674224, 0.17439961980329202, 0.08446747919598536, 0.04889079419459333, 0.028673931819134784, 0.028485751231888557, 0.022139314494000167, 0.019610488554509038, 0.022670968327361384, 0.7528137443926212, 0.44708513090102103, 0.18170500792288893, 0.08592757493056133, 0.05419807280610525, 0.038399553057713774, 0.02916834661189273, 0.03156027168602324, 0.02520153192745297, 0.014478552184906299, 0.7593230960488103, 0.4583204512537704, 0.1798806380053309, 0.07914988489574767, 0.061661366154092616, 0.03990013401911343, 0.029622097664863633, 0.022181868334724644, 0.024492408600053136, 0.020559579875359703, 0.000112126708960901, 0.05519025980085062, 0.7957925062676312, 0.23730675542648433, 0.09223246850204328, 0.043998229186286784, 0.027863133367541433, 0.014889453423247682, 0.02188350526201771, 0.02165890440158795, 0.00011983069313786354, 0.0683734002913776, 0.7920582269656165, 0.23812035444286195, 0.09501781467023535, 0.055787774034033635, 0.03479009412058926, 0.02408347101566691, 0.01756159736494318, 0.01926785137752688, 0.00034039839101365353, 0.06760356599052252, 0.7986625627672839, 0.24171238612503604, 0.09571862850630519, 0.06075823379651302, 0.04067370107109011, 0.022659344674260975, 0.022933047034696244, 0.0186486054439949, 0.0002436275482306102, 0.06355199146765221, 0.8002345248074316, 0.24390920899777424, 0.10473036836761712, 0.05233760714652162, 0.03512318699256165, 0.024286343864011413, 0.024483460874593382, 0.016283974856564964, 0.00021389707794775817, 0.04896373754761804, 0.7971625090793947, 0.24385944158076533, 0.09660697179468433, 0.06672472494746459, 0.04131875832983596, 0.019715366734478646, 0.02383013849412939, 0.018317376920260912, 1.7838146773483454e-05, 0.06780029663059853, 0.8092087462976021, 0.23580796635646809, 0.10160784618435739, 0.06486233994001817, 0.043631017603765386, 0.025441338868745313, 0.015588843874054717, 0.020144779988236206, 0.00028809852110287394, 0.055164282835398606, 0.7909220614272899, 0.23736643771618318, 0.11020434089831439, 0.05822831487596129, 0.0426682161601212, 0.016425321918162653, 0.01769960256186984, 0.01451993690083011, 2.9421806329341312e-06, 0.07399308147899293, 0.7982143610105351, 0.23756539936977542, 0.09900177567312185, 0.05768471762697158, 0.04122081009132899, 0.02285923686993853, 0.026810569646557515, 0.02137770751532116, 7.146007346435532e-05, 0.07687657293772888, 0.7990099024869082, 0.2494830328511793, 0.11084704726605439, 0.05444517295870465, 0.036352385997642046, 0.020633367687991205, 0.023281767043439205, 0.021960850928233907, 8.310624736914752e-05, 0.0695088425838532, 0.8018657295937678, 0.2430797205907489, 0.0984743447570279, 0.05995486936147348, 0.0396243859537616, 0.0210344240435303, 0.02366706194414542, 0.017334471871386627, 2.4885639565745928e-09, 0.05774064688481293, 0.794243930881071, 0.2476621711039101, 0.10005494071510299, 0.06457609293349043, 0.042731405374976335, 0.018398401606132698, 0.02501964803438109, 0.016070470264728798, 0.6303052390495203, 0.5049428116982598, 0.3527284308827287, 0.2780043362260343, 0.21814759535322328, 0.17451526906110149, 0.1573135307349071, 0.13202004306283177, 0.10665025394901866, 0.1113870727622009, 0.6212792026509264, 0.5041497960863872, 0.348578429391289, 0.27519829453855466, 0.21893311891851927, 0.18211180894972492, 0.14842887923478107, 0.13348535219953522, 0.12258133613586408, 0.10598749426875258, 0.6250758845653456, 0.5080906682880738, 0.33724581195301795, 0.24889872043420083, 0.20535025272160343, 0.17273252747440404, 0.14090238111544898, 0.131057660087271, 0.11250901087531998, 0.10135749765779895, 0.6308174333533856, 0.5112701727784376, 0.3523365318268234, 0.2574801270064406, 0.20156912016165063, 0.17400554185003572, 0.1477029498645256, 0.13742158033557353, 0.10920646475144466, 0.11083936162733897, 0.6258209578530124, 0.5178828791508282, 0.3328354839540395, 0.24752354565282098, 0.2146689747139598, 0.18099675124179584, 0.15919417565404997, 0.12383651756017174, 0.11924265449473043, 0.10126059693965729, 0.5972240287329843, 0.5116040552064477, 0.346532054088338, 0.2641520819699956, 0.20940425631894424, 0.17387029030583884, 0.16004443031385496, 0.1355348457222575, 0.10837409942441711, 0.11106362350534298, 0.6385215945332159, 0.5090101772636723, 0.3405112961037528, 0.2650556240867949, 0.20199624380123307, 0.17766435339994402, 0.15164974086068514, 0.12657298663745492, 0.11843512005141058, 0.10343022138274455, 0.6163582088646927, 0.5020309960251981, 0.35098986036428353, 0.26808478107452166, 0.20476782443282002, 0.16444627181471366, 0.147775798926299, 0.13535583825157743, 0.11897613149830252, 0.10887560104033081, 0.6174155844784053, 0.5060088644784427, 0.3555898311833097, 0.2618080922451175, 0.20584691409796976, 0.16829719023300124, 0.1516478451797869, 0.13377389985434812, 0.1151094349320155, 0.10621205122400997, 0.6170206172520109, 0.5127822201742023, 0.35611684068081745, 0.2725817979494133, 0.21236370117728987, 0.17552075500957537, 0.15312661098735839, 0.13682571547753497, 0.12234714132516511, 0.10078293654737863, 0.621435832675427, 0.5093197704616638, 0.34168253296303724, 0.25932718233769775, 0.20865823929003094, 0.173496565023406, 0.14965342212906724, 0.12937293868462046, 0.1157744274656134, 0.10488230363236023, 0.021229474934081904, 0.018440223784862743, 0.015719410606086115, 0.06515288790517736, 0.10434703993299564, 0.10478659091184678, 0.08343053367855369, 0.0966403663212739, 0.07533359263917527, 0.09167610173804469])
new_xbest = np.array([new_xbest_scaled[i] * u_bounds[i] + l_bounds[i] for i in range(len(new_xbest_scaled))])

parameters0_genome.shape

(340,)

In [4]:
costfun = prepare_stopping_costfun(ts, (11, 10), ((0.0, 20.0), (10.0, 100.0)), yms)

In [5]:
print(costfun(parameters0_genome))
costfun(new_xbest)
# RK23 cost function time: >3m, and fitness value: -
# Radau cost function time: 5.0s, and fitness value (for xbest): 0.16824699014039948
# BDF cost function time: 2.4s, and fitness value (for xbest): 0.1675684277892382

0.3440349750551145


0.07380866122769256

In [6]:
randoms = np.random.uniform(l_bounds, u_bounds, (10, len(l_bounds)))
fitnesses = np.array([costfun(random) for random in randoms])
fitnesses

array([9.77960100e+08, 5.01237601e+08, 4.87279855e+09, 8.85496836e+09,
       1.30230829e+03, 1.07649029e+03, 1.28053398e+08, 5.98849250e+07,
       1.49139704e+03, 2.14655720e+09])

----------
## Local optimization methods

#### CMA-ES optimization

In [ ]:
import dill as pkl

In [ ]:
opts = {'bounds': [l_bounds, u_bounds], 'CMA_elitist': "initial", "popsize_factor": 2, 'maxfevals':100}
xopt, es = cma.fmin2(costfun, parameters0_genome, 2.0e-3, eval_initial_x=True, options=opts)

In [ ]:
scaled_costfun = cma.ScaleCoordinates(costfun, multipliers=u_bounds)
scaled_x0 = np.array([parameters0_genome[i] / u_bounds[i] for i in range(len(parameters0_genome))])
opts = {'bounds': [[0.0 for _ in l_bounds], [1.0 for _ in u_bounds]], 'CMA_elitist': "initial", "popsize_factor": 2, 'maxfevals':30000}
xopt, es = cma.fmin2(scaled_costfun, new_xbest_scaled, 1.0e-3, eval_initial_x=True, options=opts)

(21_w,42)-aCMA-ES (mu_w=11.8,w_1=16%) in dimension 340 (seed=788716, Tue May 14 16:00:39 2024)
NOTE (module=cma, class=CMAEvolutionStrategy, method=tell, iteration=1):  initial solution injected 0.073809<0.114582
Iterat #Fevals   function value  axis ratio  sigma  min&max std  t[m:s]
    1     43 1.145821698930836e-01 1.0e+00 9.72e-04  1e-03  1e-03 0:53.7
NOTE (module=cma, class=CMAEvolutionStrategy, method=tell, iteration=2):  initial solution injected 0.073809<0.109342
    2     85 1.093423038173591e-01 1.0e+00 9.46e-04  9e-04  9e-04 1:43.9
NOTE (module=cma, class=CMAEvolutionStrategy, method=tell, iteration=3):  initial solution injected 0.073809<0.106430
    3    127 1.064301245296569e-01 1.0e+00 9.22e-04  9e-04  9e-04 2:34.3
NOTE (module=cma, class=CMAEvolutionStrategy, method=tell, iteration=4):  initial solution injected 0.073809<0.099566
    4    169 9.956584264350539e-02 1.0e+00 9.00e-04  9e-04  9e-04 3:22.6
NOTE (module=cma, class=CMAEvolutionStrategy, method=tell, iteration=

-----------------
#### Powell method optimization

In [ ]:
from scipy.optimize import rosen, OptimizeResult, Bounds, show_options

bounds = Bounds(l_bounds, u_bounds)

temp_x = []
temp_y = []

def callback(intermediate_result: np.ndarray) -> bool:
    x = intermediate_result
    y = costfun(x)
    temp_x.append(x)
    temp_y.append(y)
    print(f"Iteration {len(temp_x)}: {y}")
    return False

In [ ]:
temp_x = []
temp_y = []
res = minimize(costfun, parameters0_genome, method='Powell', bounds=bounds, callback=callback, options={'maxfev': 10000})

KeyboardInterrupt: 

In [ ]:
pkl.dump([temp_x,temp_y], open('results/F2_lbfgsb_x_run{run}.pkl'.format(run=1), 'wb'))

-----------------
#### L-BFGS-B method optimization

In [ ]:
temp_x = []
temp_y = []
res = minimize(costfun, parameters0_genome, method='L-BFGS-B', bounds=bounds, callback=callback, options={'maxfun': 10000})

In [ ]:
pkl.dump([temp_x,temp_y], open('results/F2_powell_x_run{run}.pkl'.format(run=1), 'wb'))

-----------------
#### Adam optimization

-----------------
## Global optimization methods

#### HMS optimization

In [9]:
from pyhms import minimize
import numpy as np

bounds = np.array([(l_bounds[i], u_bounds[i]) for i in range(len(l_bounds))])
solution = minimize(
    fun=costfun,
    bounds=bounds,
    maxfun=10000,
    log_level="debug"
)

2024-05-27 16:18:01 [debug    ] Starting HMS                   gsc=SingularProblemEvalLimitReached(limit=10000) height=2 levels=[EALevelConfig({'problem': <pyhms.core.problem.EvalCutoffProblem object at 0x00000236A60CFA50>, 'lsc': <pyhms.stop_conditions.usc.DontStop object at 0x00000236A5C9BD10>, 'ea_class': <class 'pyhms.demes.single_pop_eas.sea.SEA'>, 'pop_size': 690, 'generations': 1, 'sample_std_dev': 1.0, 'mutation_std': 32.5514705882353}), CMALevelConfig({'problem': <pyhms.core.problem.EvalCutoffProblem object at 0x00000236A60CFA50>, 'lsc': <pyhms.stop_conditions.lsc.FitnessSteadiness object at 0x000002369EFCF190>, 'sigma0': None, 'generations': 20})] options={'random_seed': None, 'log_level': <LoggingLevel.DEBUG: 'debug'>}
2024-05-27 17:05:16 [debug    ] Sprouted new child             id=0 metaepoch=1 seed=array([1.76972568e+02, 6.93096662e+01, 1.46667006e+02, 7.60139635e-01,
       1.88725562e+02, 1.01030102e+02, 1.25727959e+02, 1.85599469e+02,
       1.44529165e+01, 1.37965147

c:\Users\Hubert-Praca\Projects\OPUS_Optimization\F2_diff_solver.py:89: RuntimeWarning: divide by zero encountered in divide
  return [x[1], ((-x[0] - a1_int(xu) * x[1] + k_int(xu) * u) / a2_int(xu))[0]]


2024-05-27 18:19:52 [info     ] Metaepoch finished             best_fitness=401.2579527512867 best_individual=array([1.49225837e+02, 1.39274906e+01, 1.60807229e+02, 3.51231934e+01,
       1.44578436e+02, 6.99467891e+01, 1.97689968e+02, 5.44770471e+01,
       4.01967207e+01, 1.22417178e+02, 1.98138833e+02, 5.47366845e+01,
       1.77338304e+02, 3.96456890e+00, 1.82553238e+01, 1.52875093e+01,
       7.80795965e+00, 1.39952038e+02, 1.22052963e+02, 4.97086521e+01,
       1.66916084e+02, 1.08930575e+02, 1.38617439e+02, 8.60098029e+01,
       1.57726944e+02, 3.38413776e+01, 7.87444208e+01, 6.32229004e+01,
       6.14690508e+01, 1.34117784e+02, 1.48440455e+02, 6.73892899e+01,
       1.71250049e+02, 1.28330424e+02, 8.46879262e+01, 8.47163766e+01,
       6.43592189e+01, 1.48600490e+02, 5.47663957e+01, 9.72674639e+00,
       1.60516402e+02, 6.19655687e+01, 1.37800972e+02, 1.58059047e+02,
       1.46460637e+01, 1.58598761e+02, 9.31345592e+01, 1.63859426e+01,
       1.89652961e+02, 1.55690306e+02,

##### HMS optimization with 2 surrogate

In [7]:
from pyhms.config import DEFAULT_OPTIONS, BaseLevelConfig, CMALevelConfig, EALevelConfig, Options, TreeConfig
from pyhms.core.problem import EvalCutoffProblem, FunctionProblem
from pyhms.demes.single_pop_eas.sea import SEA
from pyhms.logging_ import LoggingLevel, parse_log_level
from pyhms.sprout import get_NBC_sprout
from pyhms.stop_conditions import (
    DontStop,
    FitnessSteadiness,
    GlobalStopCondition,
    MetaepochLimit,
    SingularProblemEvalLimitReached,
    UniversalStopCondition,
)
from pyhms.tree import DemeTree
from pyhms.utils.parameter_initializer import get_default_generations, get_default_mutation_std, get_default_population_size

In [8]:
low_granularity_costfun = prepare_stopping_costfun(ts, (5, 4), ((0.0, 20.0), (10.0, 100.0)), yms)

In [9]:
bounds = np.array([(l_bounds[i], u_bounds[i]) for i in range(len(l_bounds))])
function_problem_low = FunctionProblem(low_granularity_costfun, maximize=False, bounds=bounds)
function_problem = FunctionProblem(costfun, maximize=False, bounds=bounds)
wrapped_function_problem_low = EvalCutoffProblem(function_problem_low, eval_cutoff=10000)
wrapped_function_problem = EvalCutoffProblem(function_problem, eval_cutoff=10000)

gsc: GlobalStopCondition | UniversalStopCondition = (
    MetaepochLimit(50)
)
level_config = [
    EALevelConfig(
        ea_class=SEA,
        generations=get_default_generations(bounds, tree_level=0),
        problem=wrapped_function_problem_low,
        pop_size=get_default_population_size(bounds, tree_level=0),
        mutation_std=get_default_mutation_std(bounds, tree_level=0),
        lsc=DontStop(),
    ),
    CMALevelConfig(
        generations=get_default_generations(bounds, tree_level=1),
        problem=wrapped_function_problem,
        sigma0=0.01,
        lsc=FitnessSteadiness(),
    ),
]
sprout_condition = get_NBC_sprout()
options: Options = {"log_level": parse_log_level(LoggingLevel.INFO)}
config = TreeConfig(level_config, gsc, sprout_condition, options=options)
hms_tree = DemeTree(config)
hms_tree.run()
OptimizeResult(
    x=hms_tree.best_individual.genome,
    nfev=hms_tree.n_evaluations,
    fun=hms_tree.best_individual.fitness,
    nit=hms_tree.metaepoch_count,
)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (1380,) + inhomogeneous part.